# 06_Unsupervised_Model_Development_LDA

In [30]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation
import polars as pl
import joblib
from gensim.corpora import Dictionary


In [31]:
# Load the parquet file
INPUT_PATH = '../data/processed/clustered_narratives.parquet'
df_narratives = pd.read_parquet(
    INPUT_PATH,
    columns=[
        'Complaint ID',
        'processed_narrative'
    ]
)

In [32]:
df_narratives.head()

,Complaint ID,processed_narrative
0,3442136,claimed delivered package address never receiv...
1,3601853,got brink money pre paid card mail assuming un...
2,3300820,called creditor nelson cruz associate claimed ...
3,3739698,around opened credit card account online capit...
4,3285243,equifax sent credit card suggestion help impro...


In [33]:
print(len(df_narratives))

398004


In [34]:
# # Company stopwords
# company_stopwords = { 'goldman', 'sachs', 'one', 'chase', 'well', 'fargo', 'capital', 'america', 'citibank', 'citi', 'jpmorgan', 'navy', 'federal', 'union'}

In [35]:
# Run Count Vectorizer on 50k sample to save memory
df_lda_sample = df_narratives.sample(
    n=50_000,
    random_state=42
)

count_vectorizer = CountVectorizer(
    min_df=20,
    max_df=0.90,
    max_features=10_000,
    # stop_words=list(company_stopwords)
)

X_count = count_vectorizer.fit_transform(
    df_lda_sample['processed_narrative']
)

print(X_count.shape)

(50000, 6299)


In [36]:
# Run LDA on sample
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method='online',
    random_state=42,
    n_jobs=-1,
    batch_size=2048
)

lda.fit(X_count)

,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'online'
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",2048
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use in the E-step.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1


In [37]:
# Print top terms in the LDA topics
feature_names = np.array(count_vectorizer.get_feature_names_out())

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[::-1][:15]
    top_terms = feature_names[top_indices]

    print(f'\nTopic {topic_idx}')
    print(', '.join(top_terms))


Topic 0
account, issue, financial, complaint, despite, bank, fund, request, regarding, transaction, customer, matter, service, year, resolution

Topic 1
payment, fee, balance, interest, account, late, pay, month, statement, charge, paid, due, made, amount, charged

Topic 2
account, bank, money, transaction, back, america, day, claim, cash, told, atm, card, time, fund, said

Topic 3
mortgage, loan, insurance, payment, home, escrow, property, tax, year, document, company, received, foreclosure, servicing, letter

Topic 4
account, check, bank, fund, day, would, deposit, told, received, checking, branch, closed, called, call, transfer

Topic 5
charge, dispute, card, well, fargo, claim, merchant, transaction, received, refund, credit, purchase, made, never, letter

Topic 6
credit, card, account, closed, report, limit, score, received, never, application, year, bank, applied, purchase, reward

Topic 7
consumer, credit, information, act, reporting, law, account, violation, federal, right, de

In [38]:
# Add dominant topic to the sample dataframe
topic_probs = lda.transform(X_count)

df_lda_sample['dominant_topic'] = topic_probs.argmax(axis=1)
df_lda_sample['topic_probability'] = topic_probs.max(axis=1)

df_lda_sample['dominant_topic'].value_counts().sort_index()

dominant_topic
0    3120
1    5913
2    6946
3    2463
4    7631
5    6028
6    4226
7    2128
8    8139
9    3406
Name: count, dtype: int64

In [39]:
df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability
102450,7148773,complain open bank texas day ago received regu...,9,0.383484
78236,5432970,well fargo credit card file police report repo...,5,0.651494
384187,3850335,unresolved issue citibank lasted two month sti...,4,0.590685
272901,12076787,checked account today noticed encoding error c...,4,0.337794
382568,7974309,account saving checking account closed know st...,9,0.541088


In [40]:
# for topic in sorted(df_lda_sample['dominant_topic'].unique()):
#     print(f'\n===== Topic {topic} =====')
#
#     samples = (
#         df_lda_sample[df_lda_sample['dominant_topic'] == topic]
#         .sample(n=3, random_state=42)
#     )
#
#     for text in samples['processed_narrative']:
#         print('\n', text[:500])

In [41]:
# Load in cleaned narratives for the Complaint ID's in the LDA 50k sample
sample_ids = df_lda_sample['Complaint ID'].to_list()

df_readable = (
    pl.scan_parquet(INPUT_PATH)
    .filter(pl.col('Complaint ID').is_in(sample_ids))
    .select([
        'Complaint ID',
        'cleaned_consumer_narrative'
    ])
    .collect()
    .to_pandas()
)

df_lda_sample = df_lda_sample.merge(
    df_readable,
    on='Complaint ID',
    how='left'
)

In [42]:
print(df_lda_sample.columns.tolist())

['Complaint ID', 'processed_narrative', 'dominant_topic', 'topic_probability', 'cleaned_consumer_narrative']


In [43]:
# Print and inspect example narratives in each topic
for topic in sorted(df_lda_sample['dominant_topic'].unique()):
    print(f'\n===== Topic {topic} =====')

    samples = (
        df_lda_sample[df_lda_sample['dominant_topic'] == topic]
        .sample(n=3, random_state=42)
    )

    for text in samples['cleaned_consumer_narrative']:
        print('\n', text[:500])


===== Topic 0 =====

 Dear CFPB Team, I am writing to request an escalation of my unresolved disputes with Chase Bank concerning payments made to REDACTED REDACTED through REDACTED for undelivered services. Chase has rejected my disputes without providing a reasonable explanation, despite clear evidence showing that REDACTED REDACTED failed to deliver the contracted mobile app project. I have made multiple attempts to resolve this issue directly with both Chase and the merchant but have not received any satisfactory 

 Subject : Formal Complaint Regarding Fraudulent Transactions Dear SoFi I am writing to formally lodge a complaint regarding REDACTED unauthorized and fraudulent transactions that were made on REDACTED / REDACTED /year> from my SoFi Bank account by REDACTED . The details of these transactions are as follows : Transaction ID : REDACTED Amount : {$19.00} Transaction ID : REDACTED Amount : {$22.00} Both transactions were processed on the same day at REDACTED , terminal loca

In [44]:
tokenized_docs = (
    df_lda_sample['processed_narrative']
    .str.split()
    .tolist()
)

dictionary = Dictionary(tokenized_docs)

In [45]:
topics = []

for topic in lda.components_:

    top_indices = topic.argsort()[::-1][:15]

    topics.append(
        [feature_names[i] for i in top_indices]
    )

In [46]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topics,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print(f'Coherence Score: {coherence_score:.4f}')

Coherence Score: 0.4818


In [47]:
# Add Topic Probabilities as a Feature
topic_features = pd.DataFrame(
    topic_probs,
    columns=[
        f'topic_{i}_prob'
        for i in range(topic_probs.shape[1])
    ]
)

df_lda_sample = pd.concat(
    [df_lda_sample.reset_index(drop=True),
     topic_features.reset_index(drop=True)],
    axis=1
)

df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability,cleaned_consumer_narrative,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob
0,7148773,complain open bank texas day ago received regu...,9,0.383484,I have a complain open No REDACTED against Ban...,0.000971,0.000971,0.000971,0.113284,0.000971,0.000971,0.000971,0.114072,0.383333,0.383484
1,5432970,well fargo credit card file police report repo...,5,0.651494,Wells Fargo credit card {$3500.00} REDACTED / ...,0.005883,0.005883,0.005884,0.005883,0.005883,0.651494,0.005885,0.005883,0.005884,0.301438
2,3850335,unresolved issue citibank lasted two month sti...,4,0.590685,My unresolved issues with CitiBank have now la...,0.025169,0.000198,0.000198,0.000198,0.590685,0.057723,0.112084,0.000198,0.162737,0.050810
3,12076787,checked account today noticed encoding error c...,4,0.337794,I checked my account today and noticed that th...,0.000893,0.139639,0.301211,0.000893,0.337794,0.215998,0.000893,0.000893,0.000893,0.000893
4,7974309,account saving checking account closed know st...,9,0.541088,My account savings and checking account was cl...,0.006251,0.006251,0.006252,0.006251,0.408902,0.006252,0.006251,0.006251,0.006251,0.541088


In [48]:
# Identify remaining records outside of initial LDA sample
remaining_mask = ~df_narratives['Complaint ID'].isin(
    df_lda_sample['Complaint ID']
)

df_lda_remaining = df_narratives.loc[
    remaining_mask,
    ['Complaint ID', 'processed_narrative']
].copy()

print(f'Remaining records: {len(df_lda_remaining):,}')

Remaining records: 348,004


In [49]:
# Apply LDA to the remaining records without re-training
batch_size = 50_000
feature_batches = []

for start in range(0, len(df_lda_remaining), batch_size):
    end = min(start + batch_size, len(df_lda_remaining))

    print(f'Processing rows {start:,} to {end:,}...')

    batch = df_lda_remaining.iloc[start:end]

    X_batch = count_vectorizer.transform(
        batch['processed_narrative']
    )

    topic_probs = lda.transform(X_batch)

    topic_features = pd.DataFrame(
        topic_probs,
        columns=[
            f'topic_{i}_prob'
            for i in range(topic_probs.shape[1])
        ]
    )

    topic_features['Complaint ID'] = batch['Complaint ID'].values
    topic_features['dominant_topic'] = topic_probs.argmax(axis=1)

    feature_batches.append(topic_features)

Processing rows 0 to 50,000...
Processing rows 50,000 to 100,000...
Processing rows 100,000 to 150,000...
Processing rows 150,000 to 200,000...
Processing rows 200,000 to 250,000...
Processing rows 250,000 to 300,000...
Processing rows 300,000 to 348,004...


In [50]:
# Write fitted Count Vectorizer and LDA to file
joblib.dump(
    count_vectorizer,
    'count_vectorizer.pkl'
)

joblib.dump(
    lda,
    'lda_model.pkl'
)

['lda_model.pkl']

In [51]:
#Concatenate the rows
lda_features_remaining = pd.concat(
    feature_batches,
    ignore_index=True
)

# Optional: put Complaint ID first
cols = (
    ['Complaint ID', 'dominant_topic']
    + [
        col for col in lda_features_remaining.columns
        if col.startswith('topic_')
    ]
)

lda_features_remaining = lda_features_remaining[cols]

In [52]:
lda_features_remaining.head()

,Complaint ID,dominant_topic,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob
0,3442136,5,0.001205,0.021625,0.001205,0.001205,0.105470,0.837037,0.028637,0.001205,0.001205,0.001205
1,3601853,8,0.004349,0.004349,0.004350,0.205481,0.004349,0.004349,0.246325,0.004348,0.517751,0.004349
2,3300820,8,0.001852,0.001852,0.001852,0.001852,0.001852,0.001852,0.149823,0.130912,0.421510,0.286641
3,3739698,8,0.001075,0.001075,0.099669,0.001075,0.001076,0.001076,0.138743,0.001075,0.425832,0.329303
4,3285243,6,0.004167,0.004168,0.004168,0.004167,0.004168,0.004168,0.829646,0.004167,0.137015,0.004167


In [53]:
# Write to parquet
OUTPUT_PATH = '../data/processed/lda_features.parquet'

lda_features_remaining.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(lda_features_remaining.shape)

(348004, 12)
